# 02 — Data Preprocessing

Clean data, create 3-class risk labels, engineer features, scale, split, and apply SMOTE.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

## 1. Load Raw Data

In [ ]:
from src.data_loader import load_diabetes, load_heart

df_diabetes = load_diabetes()
df_heart = load_heart()
print(f'Diabetes: {df_diabetes.shape}, Heart: {df_heart.shape}')

## 2. Handle Missing Values (Zeros)

Replace biologically impossible zeros with median values.

In [ ]:
from src.preprocessing import replace_zero_with_median, DIABETES_ZERO_INVALID

# Before
print('Zero counts BEFORE cleaning:')
print((df_diabetes[DIABETES_ZERO_INVALID] == 0).sum())

df_diabetes = replace_zero_with_median(df_diabetes, DIABETES_ZERO_INVALID)

# After
print('\nZero counts AFTER cleaning:')
print((df_diabetes[DIABETES_ZERO_INVALID] == 0).sum())

## 3. Create 3-Class Risk Labels

In [ ]:
from src.preprocessing import create_risk_labels_diabetes, create_risk_labels_heart, RISK_LABEL_MAP

# Diabetes
df_diabetes['RiskLevel'] = create_risk_labels_diabetes(df_diabetes)
print('Diabetes risk distribution:')
print(df_diabetes['RiskLevel'].map(RISK_LABEL_MAP).value_counts())

# Heart
df_heart['RiskLevel'] = create_risk_labels_heart(df_heart)
print('\nHeart disease risk distribution:')
print(df_heart['RiskLevel'].map(RISK_LABEL_MAP).value_counts())

In [ ]:
# Visualize risk distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_diabetes['RiskLevel'].map(RISK_LABEL_MAP).value_counts().plot(
    kind='bar', color=['#28a745', '#ffc107', '#dc3545'], ax=axes[0])
axes[0].set_title('Diabetes Risk Distribution')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df_heart['RiskLevel'].map(RISK_LABEL_MAP).value_counts().plot(
    kind='bar', color=['#28a745', '#ffc107', '#dc3545'], ax=axes[1])
axes[1].set_title('Heart Disease Risk Distribution')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
from src.feature_engineering import (
    add_diabetes_features, add_heart_features,
    get_feature_columns_diabetes, get_feature_columns_heart,
)

df_diabetes = add_diabetes_features(df_diabetes)
df_heart = add_heart_features(df_heart)

print('Diabetes features:', get_feature_columns_diabetes())
print('\nHeart features:', get_feature_columns_heart())

In [ ]:
# Check new features
df_diabetes[['GlucoseBMI', 'AgeRisk', 'InsulinLog', 'BPCategory']].describe()

## 5. Train/Test Split & Scaling

In [ ]:
from src.preprocessing import split_data, scale_features, apply_smote
from src.model import save_scaler

# --- Diabetes ---
X_diab = df_diabetes[get_feature_columns_diabetes()]
y_diab = df_diabetes['RiskLevel']

X_train_d, X_test_d, y_train_d, y_test_d = split_data(X_diab, y_diab)
X_train_d, X_test_d, scaler_d = scale_features(X_train_d, X_test_d)
save_scaler(scaler_d, 'diabetes')

print(f'Diabetes — Train: {X_train_d.shape}, Test: {X_test_d.shape}')
print(f'Train class distribution:\n{y_train_d.value_counts().sort_index()}')

In [ ]:
# --- Heart ---
X_heart = df_heart[get_feature_columns_heart()]
y_heart = df_heart['RiskLevel']

X_train_h, X_test_h, y_train_h, y_test_h = split_data(X_heart, y_heart)
X_train_h, X_test_h, scaler_h = scale_features(X_train_h, X_test_h)
save_scaler(scaler_h, 'heart')

print(f'Heart — Train: {X_train_h.shape}, Test: {X_test_h.shape}')
print(f'Train class distribution:\n{y_train_h.value_counts().sort_index()}')

## 6. SMOTE (Class Balancing)

Apply SMOTE to **training set only** — never to test set.

In [ ]:
# Diabetes
X_train_d_sm, y_train_d_sm = apply_smote(X_train_d, y_train_d)
print(f'Diabetes AFTER SMOTE — Train: {X_train_d_sm.shape}')
print(f'Class distribution:\n{y_train_d_sm.value_counts().sort_index()}')

# Heart
X_train_h_sm, y_train_h_sm = apply_smote(X_train_h, y_train_h)
print(f'\nHeart AFTER SMOTE — Train: {X_train_h_sm.shape}')
print(f'Class distribution:\n{y_train_h_sm.value_counts().sort_index()}')

## 7. Save Processed Data

In [ ]:
from pathlib import Path

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Save for use in training notebooks
X_train_d_sm.to_csv(processed_dir / 'diabetes_X_train.csv', index=False)
X_test_d.to_csv(processed_dir / 'diabetes_X_test.csv', index=False)
y_train_d_sm.to_csv(processed_dir / 'diabetes_y_train.csv', index=False)
y_test_d.to_csv(processed_dir / 'diabetes_y_test.csv', index=False)

X_train_h_sm.to_csv(processed_dir / 'heart_X_train.csv', index=False)
X_test_h.to_csv(processed_dir / 'heart_X_test.csv', index=False)
y_train_h_sm.to_csv(processed_dir / 'heart_y_train.csv', index=False)
y_test_h.to_csv(processed_dir / 'heart_y_test.csv', index=False)

print('All processed data saved to data/processed/')